Bibliotecas Necessárias

In [1]:
# Biblioteca que possibilita que o python manipule o sistema de arquivos
# do sistema operacional. Também vamos utilizar essa função para acessar
# as variáveis de ambiente que possuem as nossas informações de autenticação.
import os

# Biblioteca que tem como objetivo realizar requisições a sistemas e APIs
import requests

# Função do módulo auth da biblioteca requests que tem como objetivo realizar a
# autenticação do usuário em um sistema ou API. 
from requests.auth import HTTPBasicAuth

# Função da biblioteca dotenv que tem como objetivo carregar no código
# as variáveis de ambiente que serão acessadas pela biblioteca os.
from dotenv import load_dotenv

# Biblioteca que possibilita a manipulação de datas e horas
import datetime

# Biblioteca que possibilita a manipulação de dados para análise
import pandas as pd

Carregando variáveis de ambientes no código

In [2]:
# Função que ira carregar as variáveis de ambiente
load_dotenv()

# getenv: Função da biblioteca os que tem como objetivo acessar as variáveis de ambientes
# carregadas pelo load_dotenv. A função recebe como argumento o nome da variável que será
# acessada.
JIRA_URL = os.getenv("JIRA_URL")

EMAIL = os.getenv("EMAIL")

SENHA_API = os.getenv("SENHA_API")

Criando a pasta de logs de erros de automação

In [3]:
# makedirs: Função da biblioteca os que tem como objetivo criar diretórios no sistema
# arquivos do sistema operacional. A função recebe como argumento o caminho da criação
# do diretório e o exist_ok com um valor booleano que indica se o código deve ignorar
# ou não o trecho de criação de diretórios (é importante ignorar a criação caso o diretório
# ja exista). 
os.makedirs("Erros", exist_ok=True)

# Trecho que irá acessar e formatar o a data e a hora capturada no sistema, vamos usar essa variável
# para atribuir horário e data no nome dos arquivos com o objetivo de criar diferentes arquivos (com base
# em sua data e hora da criação) de erros. 
agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

# Caminho completo que iremos utilizar na criação de arquivos de logs
caminho_erros = f"Erros/erros_{agora}.txt"

Conexão com  a API do Jira que contém as informações dos chamados

In [4]:
# autenticação: Variável que irá conter os valores necessários para realizarmos
# a nossa autenticação no jira. 

# HTTPBasicAuth: Função que tem como objetivo realizar a nossa autenticação na API
# utilizando as nossas credenciais. A função recebe como argumento o email da nossa
# conta no jira e a senha da nossa API  
autenticacao = HTTPBasicAuth(EMAIL, SENHA_API)

# cabecalho (header): Dicionário que tem como objetivo definir o escopo (regras) que irão ditar como
# o acesso aos dados devem acontecer.
cabecalho = {

    # Define que os parametros serão passados em formato de um dicionário json
    "Accept": "application/json",

    # Define que os dados retornados devem ser no formato json.
    "Content-Type": "application/json"
}

# Url de acesso a API que contém informações da nossa conta(JIRA_URL), das regras de acesso (rest),
# da versão da API (3), da ação que iremos realizar (search ou pesquisa em português) e a linguagem de
# consulta dos dados.
url = f"{JIRA_URL}/rest/api/3/search/jql"

# Dicionário que ira conter os dados que queremos acessar e como queremos acessar.
parametros = {

    # Consulta dos dados ordenados em ordem decrescente
    "jql":"project = Suporte ORDER BY created DESC",

    # Ira indicar a primeira linha que deve ser acessada
    "startAt": 0,

    # Ira dar um limite de linhas que devem ser retornadas a cada acesso a API
    "maxResults": 100,

    # Ira conter os campos que devem ser acessados pela API 
    "fields": ["summary", "assignee","reporter","priority","status", "resolution","created",  "updated"]
    
    }


# Ira inspecionar o bloco de conexão com o objetivo de capturar possiveis erros.
try:

    # get: Função da biblioteca requests que realiza requisições a API e sites.
    # Ela ira receber como argumento todas as informações necessárias para a
    # realização da nossa consulta na API. A função ira retornar um objeto
    # que contém as informações retornadas pelo servidor (tanto positivas
    # quanto negativas).

    # url: Contém o caminho de acesso a API.
    # headers: Recebe o cabeçalho de regras de acesso aos dados
    # auth: Reecebe os dados da nossa credencial
    # params: Contém o dicionário com as informações que queremos acessar
    resposta = requests.get(
    url,
    headers = cabecalho,
    auth = autenticacao,
    params = parametros)

    # Estrutura condicional que ira verificar o codigo retornado pelo servidor
    # da API

    # Ira verificar se o código retornado é do tipo 200 (acesso bem sucedido)
    # status_code: Atributo do objeto retornado pelo get que contém o código
    # retornado pelo servidor após uma requisição.
    if resposta.status_code == 200:

        # Se essa condição for verdadeira, vamos imprimir no terminal essa
        # mensagem.
        print("Conexão bem sucedida com a API")
    
    else:

        # Caso o servidor retorne outro tipo de código, vamos iniciar o processo
        # de registro do ocorrido no arquivo de logs de erros.

        # Impressão de uma mensagem utilizando o código retornado pelo servidor.
        print(f"Falha na conexao. código retornado: {resposta.status_code}")

        # with: Garante que os arquivo sejam fechados/encerrados após a execução
        # do código com o objetivo de evitar que o arquivo fique corrompido.

        # open: Funçãoque tem como objetivo criar ou abrir arquivos ja existentes
        # no projeto. Ela recebe como argumento o caminho do arquivo que esta sendo
        # criado ou aberto, o comando que será realizado para manipula-lo e a
        # codificação que indica como o python deve interpretar os dados.

        # caminhos_erros: Caminho de criação dos arquivos

        # a: Como append que ira adicionar valores nos arquivos criados. Observação:
        # se utilizarmos o comando 'w' (write) o python sempre irá sobrescrever
        # os dados registrados na execução anterior.

        # encoding="utf-8": Codificação que suporta a utilização de caracteres especiais

        # erro: Apelido que irá representar o arquivo que iremos criar e manipular.

        # write: Função que irá registrar dados no arquivo criado.

        # text: Atributo do objeto retornado pelo get que contém o texto
        # que indica a divergência encontrada.

        # agora: Ira conter a data e a hora da requisição a API.

        # status_code: Atributo que contém o codigo retornado pelo servidor
        with open(f"{caminho_erros}", "a", encoding="utf-8") as erros:

            erros.write(f"Falha na conexão, código retornado: {resposta.status_code}\n")

            erros.write(f"Motivo: {resposta.text}\n")

            agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

            erros.write(f"Data e hora da Requisição: {agora}\n")

            erros.write("------------------------------------------------\n")
            
except requests.exceptions.RequestException as erro:

    # Exceção que ira lidar com erros na requisição a API. Ela ira repetir
    # todo o processo que documentamos anteriormente
    print(f"Falha na conexão com a API: {erro}")

    with open(f"{caminho_erros}", "a", encoding="utf-8") as erros:

            erros.write(f"Falha na conexão, código retornado: {erro}\n")

            erros.write(f"Motivo: {resposta.text}\n")

            agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

            erros.write(f"Data e hora da Requisição: {agora}\n")

            erros.write("------------------------------------------------\n")


Conexão bem sucedida com a API


Visualizando o json retornado pela API

In [5]:
# Ira conter o dicionário json que contém os dados coletados da API (dados
# do projeto que estamos querendo acessar). 

# json: Método da classe retornada pela função get que transforma a resposta
# da API (em protocolo HTTP) em um dicionário Python.
dados_api = resposta.json()

# Ira pegar a lista de dicionários da chave issues que contém as informações que queremos acessar
# (informações de cada ticket) do nosso projeto. 
issues = dados_api.get("issues")


Verificando o tipo de cada dado no dicionário de valores: Como alguns itens do
dicionário possuem outros dicionários como valor (subdicionários), é necessário
verificar os tipos de cada dado para definir a forma que iremos acessar ele.

In [6]:
# Para realizar essa ação vamos percorrer a lista de valores da chave issues e
# acessaremos a lista de valores do field (campos da tabela)
for issue in issues:

    # Ira acessar a lista de valores do fields
    fields = issue.get("fields")

    # Ira percorrer os valores do dicionário fields. A função items é um método
    # nativo do dos dicionários do python (dict). Ela serve para extrair e 
    # retornar simultaneamente as chaves e os valores de um dicionário. Quando
    # chamamos o .items() em um dicionário, ele te devolve uma visão contendo
    # tuplas no formato (chave, valor).
    for chave, campo in fields.items():

        # Ira imprimir as chave o tipo do seu valor.
        # type: Função nativa do python que tem como objetivo retornar
        # o tipo de dado passado como argumento na função. 
        print(f"Chave:{chave} | Tipo do dado: {type(campo)}")

    

Chave:summary | Tipo do dado: <class 'str'>
Chave:created | Tipo do dado: <class 'str'>
Chave:reporter | Tipo do dado: <class 'NoneType'>
Chave:assignee | Tipo do dado: <class 'NoneType'>
Chave:priority | Tipo do dado: <class 'dict'>
Chave:resolution | Tipo do dado: <class 'NoneType'>
Chave:updated | Tipo do dado: <class 'str'>
Chave:status | Tipo do dado: <class 'dict'>
Chave:summary | Tipo do dado: <class 'str'>
Chave:created | Tipo do dado: <class 'str'>
Chave:reporter | Tipo do dado: <class 'NoneType'>
Chave:assignee | Tipo do dado: <class 'NoneType'>
Chave:priority | Tipo do dado: <class 'dict'>
Chave:resolution | Tipo do dado: <class 'dict'>
Chave:updated | Tipo do dado: <class 'str'>
Chave:status | Tipo do dado: <class 'dict'>
Chave:summary | Tipo do dado: <class 'str'>
Chave:created | Tipo do dado: <class 'str'>
Chave:reporter | Tipo do dado: <class 'dict'>
Chave:assignee | Tipo do dado: <class 'NoneType'>
Chave:priority | Tipo do dado: <class 'dict'>
Chave:resolution | Tipo do

        Tipos de cada dado do dicionário field

summary: str

assignee: NoneType 

reporter: dict

priority: dict

status: dict

resolution: NoneType 

created: str

uptaded: str



criando a lista que sera transformada em DataFrame: Como será necessário visualizar os dados para trata-los, vamos transforma-los em um DataFrame com o objetivo de facilitar a visualização e manipulação dos dados usando funções
presentes na biblioteca pandas. 

In [7]:
# Ira criar a lista que conterá os valores que serão convertidos em um DataFrame.
lista_campos = []

# Ira percorrer o dicionário issues que contém os valores de cada ticket presente no jira
for issue in issues:

    # Ira acessar o subdicionário fields que contém os campos da tabela criada no jira. Vamos acessar
    # cada chave (nome do campo na tabela) com o objetivo de acessar os valores do subdicionário.
    fields = issue.get("fields")

     # Lidando com valores nulos: Ao seguir com o script eu descobri a existencia de valores nulos
    # que o pandas não estava conseguindo identificar (acredito que isso estava ocorrendo com
    # dados sem tipos/NoneType). Então resolvi criar uma estrutura condicional que verifica se há
    # ou não a presença de dados com o objetivo de adicionar um valor padrão em valores nulos.

    # Diferença nos acesso aos dados: Vale lembrar que dados do tipo texto são acessados de formas diferentes
    # de um dicionário. Por exemplo:

    # Acesso ao dado do tipo texto: coluna_acessada = fields.get(nome da chave)

    # Acesso ao dado do tipo dicionário = coluna_acessada = fields.get(chave que contem o subdicionario).get(chave do valor presente no subdicionário)

    # Ira conter o nome/descrição do ticket
    summary = fields.get("summary")

    # Ira conter o responsável pelo ticket
    assignee_obj = fields.get("assignee")

    assignee = assignee_obj.get("displayName") if assignee_obj else "Não Atribuido"

    # Ira conter o relator do ticket  
    reporter_obj = fields.get("reporter")
    
    reporter = reporter_obj.get("displayName") if reporter_obj else "Anônimo"

    # Irá conter a prioridade do ticket
    priority = fields.get("priority").get("name")

    # Ira conter o status do ticket
    status = fields.get("status").get("name")

    # Irá conter a resolução do ticket
    resolution_obj = fields.get("resolution")

    resolution = fields.get("displayName") if resolution_obj  else "Problema não resolvido" 

    # Ira conter a data e a hora da criação do ticket
    created = fields.get("created")

    # Ira conter a data e a hora da atualização do ticket
    uptaded = fields.get("updated")

    # Nessa etapa, precisamos adicionar cada valor acessado na lista, para isso vamos 
    # criar um dicionário dentro da lista que ira conter todos os dados acessados
    # anteriormente.Vou aproveitar a oportunidade para renomear o nome das colunas 
    # através da definição dos nomes das chaves dos dicionários.
    lista_campos.append({"ticket": summary,
                          "responsável": assignee,
                          "relator": reporter,
                          "prioridade": priority,
                          "status": status,
                          "resolucao": resolution,
                          "criado": created,
                          "atualizacao": uptaded
                          
    })


Visualizando a lista pronta

In [8]:
lista_campos

[{'ticket': 'queda da funcionalidade de login',
  'responsável': 'Não Atribuido',
  'relator': 'Anônimo',
  'prioridade': 'Medium',
  'status': 'Aberto',
  'resolucao': 'Problema não resolvido',
  'criado': '2026-10-07T12:23:01.588-0300',
  'atualizacao': '2026-10-15T13:37:59.081-0300'},
 {'ticket': 'Atualização da página de sobre',
  'responsável': 'Não Atribuido',
  'relator': 'Anônimo',
  'prioridade': 'Medium',
  'status': 'Concluída',
  'resolucao': None,
  'criado': '2026-10-07T12:10:36.365-0300',
  'atualizacao': '2026-10-16T13:00:20.572-0300'},
 {'ticket': 'Falha nas atualizações de cadastro dos clientes',
  'responsável': 'Não Atribuido',
  'relator': 'Caike Oliveira Domingues',
  'prioridade': 'Medium',
  'status': 'Aguardando pelo suporte',
  'resolucao': 'Problema não resolvido',
  'criado': '2026-10-07T11:53:36.240-0300',
  'atualizacao': '2026-10-07T11:53:36.955-0300'},
 {'ticket': 'Erro no carregamento da página de produtos',
  'responsável': 'Não Atribuido',
  'relator'

Convertendo a lista em um DataFrame

In [9]:
# DataFrame: Função da biblioteca pandas que tem como objetivo converter uma lista de dados
# em um DataFrame que possui linhas e colunas. A função retorna um objeto DataFrame que possui
# metodos e atributos de manipulação de dados.
df = pd.DataFrame(lista_campos)

Visualizando as 15 linhas de dados

In [10]:
df.head(15)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao
0,queda da funcionalidade de login,Não Atribuido,Anônimo,Medium,Aberto,Problema não resolvido,2026-10-07T12:23:01.588-0300,2026-10-15T13:37:59.081-0300
1,Atualização da página de sobre,Não Atribuido,Anônimo,Medium,Concluída,None,2026-10-07T12:10:36.365-0300,2026-10-16T13:00:20.572-0300
2,Falha nas atualizações de cadastro dos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Aguardando pelo suporte,Problema não resolvido,2026-10-07T11:53:36.240-0300,2026-10-07T11:53:36.955-0300
3,Erro no carregamento da página de produtos,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,None,2026-10-07T11:41:18.653-0300,2026-10-07T12:26:24.884-0300
4,Atualização do front-end,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06T17:27:11.997-0300,2026-10-06T17:27:12.657-0300
5,Falha no backup dos dados,Não Atribuido,Caike Oliveira Domingues,Medium,Work in progress,Problema não resolvido,2026-10-06T17:06:02.051-0300,2026-10-16T13:00:45.680-0300
6,problemas no botão de envio na tela de cadastro,Não Atribuido,Caike Oliveira Domingues,Medium,Pending,Problema não resolvido,2026-10-06T17:02:42.997-0300,2026-10-06T17:19:41.513-0300
7,Atualização de servidores,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06T16:58:13.325-0300,2026-10-06T16:58:14.079-0300
8,Falha no cadastro de novos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,None,2026-10-06T16:50:06.333-0300,2026-10-16T13:01:17.745-0300
9,Queda do Sistema,Caike Oliveira Domingues,Caike Oliveira Domingues,Medium,Aberto,Problema não resolvido,2026-10-06T16:44:55.765-0300,2026-10-07T12:27:36.099-0300


Visualizando as informações gerais da tabela

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ticket       15 non-null     object
 1   responsável  15 non-null     object
 2   relator      15 non-null     object
 3   prioridade   15 non-null     object
 4   status       15 non-null     object
 5   resolucao    9 non-null      object
 6   criado       15 non-null     object
 7   atualizacao  15 non-null     object
dtypes: object(8)
memory usage: 1.1+ KB


Como podemos observar, a coluna de resolução ainda possui dados nulos

In [12]:
# Vamos usar a função fillna que tem como objetivo substituir valores nulos
# por um valor padrão. Essa abordagem é interessante quando temos dados que
# não podem ser apagados. A função recebe como argumento o valor padrão
# que irá substituir os valores nulos.
df['resolucao'] = df['resolucao'].fillna("Atividade Encerrada")

# Verificando se a substituição foi bem sucedida
df['resolucao']

0     Problema não resolvido
1        Atividade Encerrada
2     Problema não resolvido
3        Atividade Encerrada
4     Problema não resolvido
5     Problema não resolvido
6     Problema não resolvido
7     Problema não resolvido
8        Atividade Encerrada
9     Problema não resolvido
10       Atividade Encerrada
11       Atividade Encerrada
12    Problema não resolvido
13       Atividade Encerrada
14    Problema não resolvido
Name: resolucao, dtype: object

Nessa etapa será necessário converter as colunas de criação e atualização em um formato de data e hora, pois precisaremos desse formato para calcularmos
a duração de resolução dessa tarefa (que sera um dos critérios de classificação dos dados)

In [13]:
# to_datetime: Função da biblioteca pandas que tem como objetivo converter
# valores em formato de data e hora. A função recebe como argumento a coluna
# que contém a lista de valores que será convertida (as colunas de 'criado'
# e 'atualizacao'). 
df['criado'] = pd.to_datetime(df['criado'])

df['atualizacao'] = pd.to_datetime(df['atualizacao'])

# Visualizando se a conversão funcionou
df.head(15)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao
0,queda da funcionalidade de login,Não Atribuido,Anônimo,Medium,Aberto,Problema não resolvido,2026-10-07 12:23:01.588000-03:00,2026-10-15 13:37:59.081000-03:00
1,Atualização da página de sobre,Não Atribuido,Anônimo,Medium,Concluída,Atividade Encerrada,2026-10-07 12:10:36.365000-03:00,2026-10-16 13:00:20.572000-03:00
2,Falha nas atualizações de cadastro dos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Aguardando pelo suporte,Problema não resolvido,2026-10-07 11:53:36.240000-03:00,2026-10-07 11:53:36.955000-03:00
3,Erro no carregamento da página de produtos,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-07 11:41:18.653000-03:00,2026-10-07 12:26:24.884000-03:00
4,Atualização do front-end,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 17:27:11.997000-03:00,2026-10-06 17:27:12.657000-03:00
5,Falha no backup dos dados,Não Atribuido,Caike Oliveira Domingues,Medium,Work in progress,Problema não resolvido,2026-10-06 17:06:02.051000-03:00,2026-10-16 13:00:45.680000-03:00
6,problemas no botão de envio na tela de cadastro,Não Atribuido,Caike Oliveira Domingues,Medium,Pending,Problema não resolvido,2026-10-06 17:02:42.997000-03:00,2026-10-06 17:19:41.513000-03:00
7,Atualização de servidores,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 16:58:13.325000-03:00,2026-10-06 16:58:14.079000-03:00
8,Falha no cadastro de novos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-06 16:50:06.333000-03:00,2026-10-16 13:01:17.745000-03:00
9,Queda do Sistema,Caike Oliveira Domingues,Caike Oliveira Domingues,Medium,Aberto,Problema não resolvido,2026-10-06 16:44:55.765000-03:00,2026-10-07 12:27:36.099000-03:00


Criando a coluna que ira conter a classificação dos SLA

In [14]:
# Vamos adicionar como valor padrão da coluna o valor "Em Andamento" que indicara
# que a tarefa ainda não foi concluida/cancelada, logo não há uma classificação 
# no momento.
df['SLA'] = "Em Andamento" 

df.head(15)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao,SLA
0,queda da funcionalidade de login,Não Atribuido,Anônimo,Medium,Aberto,Problema não resolvido,2026-10-07 12:23:01.588000-03:00,2026-10-15 13:37:59.081000-03:00,Em Andamento
1,Atualização da página de sobre,Não Atribuido,Anônimo,Medium,Concluída,Atividade Encerrada,2026-10-07 12:10:36.365000-03:00,2026-10-16 13:00:20.572000-03:00,Em Andamento
2,Falha nas atualizações de cadastro dos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Aguardando pelo suporte,Problema não resolvido,2026-10-07 11:53:36.240000-03:00,2026-10-07 11:53:36.955000-03:00,Em Andamento
3,Erro no carregamento da página de produtos,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-07 11:41:18.653000-03:00,2026-10-07 12:26:24.884000-03:00,Em Andamento
4,Atualização do front-end,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 17:27:11.997000-03:00,2026-10-06 17:27:12.657000-03:00,Em Andamento
5,Falha no backup dos dados,Não Atribuido,Caike Oliveira Domingues,Medium,Work in progress,Problema não resolvido,2026-10-06 17:06:02.051000-03:00,2026-10-16 13:00:45.680000-03:00,Em Andamento
6,problemas no botão de envio na tela de cadastro,Não Atribuido,Caike Oliveira Domingues,Medium,Pending,Problema não resolvido,2026-10-06 17:02:42.997000-03:00,2026-10-06 17:19:41.513000-03:00,Em Andamento
7,Atualização de servidores,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 16:58:13.325000-03:00,2026-10-06 16:58:14.079000-03:00,Em Andamento
8,Falha no cadastro de novos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-06 16:50:06.333000-03:00,2026-10-16 13:01:17.745000-03:00,Em Andamento
9,Queda do Sistema,Caike Oliveira Domingues,Caike Oliveira Domingues,Medium,Aberto,Problema não resolvido,2026-10-06 16:44:55.765000-03:00,2026-10-07 12:27:36.099000-03:00,Em Andamento


Verificando se a coluna foi criada

In [15]:
df.columns

Index(['ticket', 'responsável', 'relator', 'prioridade', 'status', 'resolucao',
       'criado', 'atualizacao', 'SLA'],
      dtype='object')

Classificando os sla: Nessa etapa, vamos criar uma função que irá classificar todas as tarefas concluidas e canceladas utilizando como critério as colunas: "criado", "atualizacao", "prioridade" e "status" com o objetivo de registrar indice de eficiência do SLA     

In [16]:
# Função que ira classificar as tarefas de acordo com a eficiência de suas
# resoluções com base em sua prioridade, tempo de resolução e status (se
# a tarefa foi concluida ou cancelada). A função recebe como argumento

# prioridade_escolhida: Ira conter a prioridade que esta sendo análisada

# limite: Ira conter a quantidade de horas limite para cada resolução da 
# tarefa (com base em sua prioridade).  

# falta_1_hora: Ira indicar quando faltar 1 hora para a resolução da tarefa
def classificar_sla(prioridade_escolhida, limite, falta_1_hora):

    # Loop que ira percorrer o dataframe com o objetivo de acessar os valores
    # do dataframe para podermos manipula-los.

    # iterrows: Função do pandas que retorna todas as linhas (e seus indices) 
    # das colunas de um Dataframe.
    
    # indice: Indice de cada linha das colunas acessadas
     
    # linha: Representa o valor da linha que esta sendo acessada.  
    for indice, linha in df.iterrows():

        # Ira acessar as linhas da coluna 'criado'
        criado = linha['criado']

        # Ira acessar as linhas da coluna 'atualizacao'
        atualizacao = linha['atualizacao']

        # Irá acessar as linhas da coluna 'status'
        status = linha['status']

        # Irá acessar as linhas da coluna 'prioridade'
        prioridade = linha['prioridade']

        # Ira calcular o intervalo de tempo da coluna de data de atualização 
        # com a coluna de data de criação.O total seconds tem como objetivo
        # acessar o total de segundos e dividi-lo por 3600 para garantir
        # que o formato do resultado do calculo (subtração das colunas)
        # seja em formato de hora e data.  
        calc_intervalo = (atualizacao - criado).total_seconds() / 3600

        # Estrutura condicional que irá classificar as tarefas em SLA. Basicamente essa 
        # estrutura ira verificar se o cálculo do intervalo esta dentro do prazo estabelecido
        # de acordo com a sua prioridade e o seu status (para classificar apenas tarefas concluidas
        # ou canceladas). Dentro do trecho, vamos acessar as linhas através dos seus indices
        # com o objetivo de atribuir a linha a classificação condinzente com a tarefa (esse
        # valor ira substituir o valor padrão "Em Andamento" que definimos na criação da coluna
        # SLA).
        if calc_intervalo <= limite and prioridade == prioridade_escolhida and (status == "Concluída" or status == "Cancelado"):

            df.loc[indice, "SLA"] = "SLA Cumprido"
        
        elif calc_intervalo == falta_1_hora and prioridade == prioridade_escolhida and (status == "Concluída" or status == "Cancelado"):

            df.loc[indice, "SLA"] = "SLA em Risco"
        
        elif calc_intervalo > limite and prioridade == prioridade_escolhida and (status == "Concluída" or status == "Cancelado"):

            df.loc[indice, "SLA"] = "SLA Estourado"

Chamada da função para classificar cada tarefa de acordo com a sua prioridade

In [17]:
classificar_sla("High", 24, 23)

df.head(15)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao,SLA
0,queda da funcionalidade de login,Não Atribuido,Anônimo,Medium,Aberto,Problema não resolvido,2026-10-07 12:23:01.588000-03:00,2026-10-15 13:37:59.081000-03:00,Em Andamento
1,Atualização da página de sobre,Não Atribuido,Anônimo,Medium,Concluída,Atividade Encerrada,2026-10-07 12:10:36.365000-03:00,2026-10-16 13:00:20.572000-03:00,Em Andamento
2,Falha nas atualizações de cadastro dos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Aguardando pelo suporte,Problema não resolvido,2026-10-07 11:53:36.240000-03:00,2026-10-07 11:53:36.955000-03:00,Em Andamento
3,Erro no carregamento da página de produtos,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-07 11:41:18.653000-03:00,2026-10-07 12:26:24.884000-03:00,Em Andamento
4,Atualização do front-end,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 17:27:11.997000-03:00,2026-10-06 17:27:12.657000-03:00,Em Andamento
5,Falha no backup dos dados,Não Atribuido,Caike Oliveira Domingues,Medium,Work in progress,Problema não resolvido,2026-10-06 17:06:02.051000-03:00,2026-10-16 13:00:45.680000-03:00,Em Andamento
6,problemas no botão de envio na tela de cadastro,Não Atribuido,Caike Oliveira Domingues,Medium,Pending,Problema não resolvido,2026-10-06 17:02:42.997000-03:00,2026-10-06 17:19:41.513000-03:00,Em Andamento
7,Atualização de servidores,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 16:58:13.325000-03:00,2026-10-06 16:58:14.079000-03:00,Em Andamento
8,Falha no cadastro de novos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-06 16:50:06.333000-03:00,2026-10-16 13:01:17.745000-03:00,Em Andamento
9,Queda do Sistema,Caike Oliveira Domingues,Caike Oliveira Domingues,Medium,Aberto,Problema não resolvido,2026-10-06 16:44:55.765000-03:00,2026-10-07 12:27:36.099000-03:00,Em Andamento


In [18]:
classificar_sla("Medium", 72, 71)

df.head(15)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao,SLA
0,queda da funcionalidade de login,Não Atribuido,Anônimo,Medium,Aberto,Problema não resolvido,2026-10-07 12:23:01.588000-03:00,2026-10-15 13:37:59.081000-03:00,Em Andamento
1,Atualização da página de sobre,Não Atribuido,Anônimo,Medium,Concluída,Atividade Encerrada,2026-10-07 12:10:36.365000-03:00,2026-10-16 13:00:20.572000-03:00,SLA Estourado
2,Falha nas atualizações de cadastro dos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Aguardando pelo suporte,Problema não resolvido,2026-10-07 11:53:36.240000-03:00,2026-10-07 11:53:36.955000-03:00,Em Andamento
3,Erro no carregamento da página de produtos,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-07 11:41:18.653000-03:00,2026-10-07 12:26:24.884000-03:00,SLA Cumprido
4,Atualização do front-end,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 17:27:11.997000-03:00,2026-10-06 17:27:12.657000-03:00,Em Andamento
5,Falha no backup dos dados,Não Atribuido,Caike Oliveira Domingues,Medium,Work in progress,Problema não resolvido,2026-10-06 17:06:02.051000-03:00,2026-10-16 13:00:45.680000-03:00,Em Andamento
6,problemas no botão de envio na tela de cadastro,Não Atribuido,Caike Oliveira Domingues,Medium,Pending,Problema não resolvido,2026-10-06 17:02:42.997000-03:00,2026-10-06 17:19:41.513000-03:00,Em Andamento
7,Atualização de servidores,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 16:58:13.325000-03:00,2026-10-06 16:58:14.079000-03:00,Em Andamento
8,Falha no cadastro de novos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-06 16:50:06.333000-03:00,2026-10-16 13:01:17.745000-03:00,SLA Estourado
9,Queda do Sistema,Caike Oliveira Domingues,Caike Oliveira Domingues,Medium,Aberto,Problema não resolvido,2026-10-06 16:44:55.765000-03:00,2026-10-07 12:27:36.099000-03:00,Em Andamento


In [19]:
classificar_sla("Low", 120, 119)
df.head(15)

,ticket,responsável,relator,prioridade,status,resolucao,criado,atualizacao,SLA
0,queda da funcionalidade de login,Não Atribuido,Anônimo,Medium,Aberto,Problema não resolvido,2026-10-07 12:23:01.588000-03:00,2026-10-15 13:37:59.081000-03:00,Em Andamento
1,Atualização da página de sobre,Não Atribuido,Anônimo,Medium,Concluída,Atividade Encerrada,2026-10-07 12:10:36.365000-03:00,2026-10-16 13:00:20.572000-03:00,SLA Estourado
2,Falha nas atualizações de cadastro dos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Aguardando pelo suporte,Problema não resolvido,2026-10-07 11:53:36.240000-03:00,2026-10-07 11:53:36.955000-03:00,Em Andamento
3,Erro no carregamento da página de produtos,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-07 11:41:18.653000-03:00,2026-10-07 12:26:24.884000-03:00,SLA Cumprido
4,Atualização do front-end,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 17:27:11.997000-03:00,2026-10-06 17:27:12.657000-03:00,Em Andamento
5,Falha no backup dos dados,Não Atribuido,Caike Oliveira Domingues,Medium,Work in progress,Problema não resolvido,2026-10-06 17:06:02.051000-03:00,2026-10-16 13:00:45.680000-03:00,Em Andamento
6,problemas no botão de envio na tela de cadastro,Não Atribuido,Caike Oliveira Domingues,Medium,Pending,Problema não resolvido,2026-10-06 17:02:42.997000-03:00,2026-10-06 17:19:41.513000-03:00,Em Andamento
7,Atualização de servidores,Não Atribuido,Caike Oliveira Domingues,Medium,Revisar,Problema não resolvido,2026-10-06 16:58:13.325000-03:00,2026-10-06 16:58:14.079000-03:00,Em Andamento
8,Falha no cadastro de novos clientes,Não Atribuido,Caike Oliveira Domingues,Medium,Concluída,Atividade Encerrada,2026-10-06 16:50:06.333000-03:00,2026-10-16 13:01:17.745000-03:00,SLA Estourado
9,Queda do Sistema,Caike Oliveira Domingues,Caike Oliveira Domingues,Medium,Aberto,Problema não resolvido,2026-10-06 16:44:55.765000-03:00,2026-10-07 12:27:36.099000-03:00,Em Andamento


Criando o relatório que será usado na construção do dash no powerbi

In [20]:
# Trecho que irá acessar e formatar o a data e a hora capturada no sistema, vamos usar essa variável
# para atribuir horário e data no nome dos arquivos com o objetivo de criar diferentes arquivos (com base
# em sua data e hora da criação) de erros. 
agora = datetime.datetime.now().strftime("%d_%m_%Y_%H_%M")

# Ira criar a pasta que armazenara os relatórios
os.makedirs("Relatorios", exist_ok=True)

# Ira remover o fuso horário das horas que podem causar erros e problemas
# na geração do arquivo excel.
df['criado'] = pd.to_datetime(df['criado']).dt.tz_localize(None)

df['atualizacao'] = pd.to_datetime(df['atualizacao']).dt.tz_localize(None)

# to_excel: Método da classe retornada pelo pandas que tem como objetivo
# criar um arquivo excel usando os dados de um DataFrame. A função recebe
# como argumento o caminho que o arquivo ficara salvo e o index=None que define
# que os indices das colunas não devem ser salvos
df.to_excel(f"Relatorios/relatorios_{agora}.xlsx", index=None)